# DORA Percentage Deployment Failure Rates

## Pre-Requisites

In [1]:
from dash import dcc
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

In [2]:
# Useful reference: <https://pandas.pydata.org/docs/getting_started/intro_tutorials/09_timeseries.html>
raw_file_path = '/home/lnx_workspaces/bpp_projects/bpp_module2_project/doraview/data/json/deployment_frequency.json'

# Try reading with explicit encoding and error handling
try:
    df_deployments = pd.read_json(raw_file_path, encoding='utf-8', convert_dates=["deployed_at"])
    print("\nDataframe info:")
    print(df_deployments.info())
    print("\nFirst 5 rows:")
    print(df_deployments.head())
except Exception as e:
    print(f"Error: {str(e)}")


Dataframe info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   deployment_id     44 non-null     object             
 1   application_id    44 non-null     object             
 2   application_name  44 non-null     object             
 3   environment       44 non-null     object             
 4   deployed_at       44 non-null     datetime64[ns, UTC]
 5   status            44 non-null     object             
 6   version           44 non-null     object             
dtypes: datetime64[ns, UTC](1), object(6)
memory usage: 2.5+ KB
None

First 5 rows:
  deployment_id application_id application_name environment  \
0   df_55eeff30         app001     Auth Service  production   
1   df_952faf50         app001     Auth Service     staging   
2   df_45056a55         app001     Auth Service  production   
3   df_4a4be47

In [3]:
# Add new row and quater columns
df_deployments["month"] = df_deployments["deployed_at"].dt.month

df_deployments.head(100)

,deployment_id,application_id,application_name,environment,deployed_at,status,version,month
0,df_55eeff30,app001,Auth Service,production,2025-08-31 02:05:00+00:00,success,v3.3.4,8
1,df_952faf50,app001,Auth Service,staging,2025-08-07 11:06:00+00:00,success,v3.7.0,8
2,df_45056a55,app001,Auth Service,production,2025-07-27 12:08:00+00:00,success,v2.0.3,7
3,df_4a4be475,app001,Auth Service,staging,2025-08-16 06:24:00+00:00,success,v3.8.4,8
4,df_9ddc9087,app001,Auth Service,staging,2025-07-29 03:29:00+00:00,success,v1.7.6,7
5,df_09f6a4a8,app001,Auth Service,production,2025-09-30 17:01:00+00:00,success,v1.3.6,9
6,df_0721d117,app001,Auth Service,production,2025-08-11 14:09:00+00:00,success,v2.1.9,8
7,df_3ad908f6,app001,Auth Service,staging,2025-08-07 03:03:00+00:00,success,v3.9.3,8
8,df_295ed3f1,app001,Auth Service,production,2025-07-07 16:35:00+00:00,success,v2.4.3,7
9,df_1df285b7,app001,Auth Service,staging,2025-08-09 09:38:00+00:00,success,v2.9.6,8


## Tinkering

In [4]:
df_fail_rates = df_deployments[['application_id','month','status']]
df_fail_rates

,application_id,month,status
0,app001,8,success
1,app001,8,success
2,app001,7,success
3,app001,8,success
4,app001,7,success
5,app001,9,success
6,app001,8,success
7,app001,8,success
8,app001,7,success
9,app001,8,success


In [6]:
df_count = df_deployments.groupby(['month','application_id','status']).agg({'status':'count'})

print(df_count)

                              status
month application_id status         
7     app001         success       6
      app002         failed        1
                     success       1
      app003         failed        1
                     success       3
8     app001         success       6
      app002         failed        1
                     success       4
      app003         failed        2
                     success       9
9     app001         success       2
      app002         failed        1
                     success       3
      app003         failed        1
                     success       3


In [10]:
df_app2 = df_count.xs('app002', level='application_id')

print(df_app2)

               status
month status         
7     failed        1
      success       1
8     failed        1
      success       4
9     failed        1
      success       3


In [8]:
df_app2['Percentage'] = 100 * df_app2['status'] / df_app2.groupby(level=0)['status'].transform('sum')



print(df_app2_indexed)

/tmp/ipykernel_10776/140018116.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_app2['Percentage'] = 100 * df_app2['status'] / df_app2.groupby(level=0)['status'].transform('sum')


ValueError: cannot insert status, already exists

In [ ]:
# display dataframe as figure
fig_month_stat_bar = px.bar(
	data_frame=df_app2,
	title="Total Deployments by Status",
	x="month",
	y="Percentage",
	color="status",
	color_discrete_map={
		"success":"#636EFA",
		"failed":"#EF553B"
		},
)

fig.update_layout(barmode='stack')

fig_month_stat_bar.update_yaxes(
	title_text="Deployment Count"
)

fig_month_stat_bar.update_xaxes(
	title_text="Deployment Month",
	tickvals=list(range(1,13)),
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
)

fig_month_stat_bar.show()

## Just app002

In [20]:
df_app2_only = df_deployments[df_deployments['application_id'] == 'app002']

# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.set_index.html
df_app2_only.reset_index(drop=True, inplace=True)

df_app2_only

,deployment_id,application_id,application_name,environment,deployed_at,status,version,month
0,df_8966166b,app002,Inventory API,production,2025-07-22 04:15:00+00:00,success,v2.2.8,7
1,df_e2070f8a,app002,Inventory API,staging,2025-08-02 05:16:00+00:00,success,v1.8.6,8
2,df_a29aae73,app002,Inventory API,production,2025-07-19 08:43:00+00:00,failed,v1.5.9,7
3,df_5fe481e2,app002,Inventory API,staging,2025-09-17 17:46:00+00:00,success,v2.0.1,9
4,df_42d14791,app002,Inventory API,staging,2025-08-02 23:31:00+00:00,success,v3.4.7,8
5,df_883f7d05,app002,Inventory API,staging,2025-08-16 16:08:00+00:00,failed,v3.0.1,8
6,df_f6f4c236,app002,Inventory API,staging,2025-08-19 03:59:00+00:00,success,v3.6.3,8
7,df_b864e01d,app002,Inventory API,staging,2025-09-03 12:34:00+00:00,success,v1.3.5,9
8,df_df253d66,app002,Inventory API,staging,2025-09-11 08:26:00+00:00,failed,v3.5.6,9
9,df_33cfc06f,app002,Inventory API,staging,2025-09-03 22:30:00+00:00,success,v1.9.4,9


In [22]:
df_app2_simple = df_app2_only[['month','status']]

df_app2_simple

,month,status
0,7,success
1,8,success
2,7,failed
3,9,success
4,8,success
5,8,failed
6,8,success
7,9,success
8,9,failed
9,9,success


In [26]:
df_app2_grouped = df_app2_simple.groupby(['month','status']).agg({'status':'count'})

print(df_app2_grouped)

               status
month status         
7     failed        1
      success       1
8     failed        1
      success       4
9     failed        1
      success       3


In [37]:
df_app2_perentage = df_app2_grouped.groupby(level=0).apply(lambda x: 100 * x / x.sum())

# Fix the index (drop the duplicate month level)
df_app2_perentage.index = df_app2_perentage.index.droplevel(1)

# Rename the column to avoid conflict during reset_index
df_app2_perentage = df_app2_perentage.rename(columns={'status': 'Percentage'})

print("df_app2_perentage:")
print(df_app2_perentage)
print("Columns:", df_app2_perentage.columns)
print("Index names:", df_app2_perentage.index.names)

# Reset index to create a DataFrame suitable for plotting
df_app2_perentage_df = df_app2_perentage.reset_index()

print(df_app2_perentage_df)

df_app2_perentage:
               Percentage
month status             
7     failed         50.0
      success        50.0
8     failed         20.0
      success        80.0
9     failed         25.0
      success        75.0
Columns: Index(['Percentage'], dtype='object')
Index names: ['month', 'status']
   month   status  Percentage
0      7   failed        50.0
1      7  success        50.0
2      8   failed        20.0
3      8  success        80.0
4      9   failed        25.0
5      9  success        75.0


In [38]:
# display dataframe as figure
fig_month_stat_bar = px.bar(
	data_frame=df_app2_perentage_df,
	title="Deployment Failure Rates by Month",
	x="month",
	y="Percentage",
	color="status",
	color_discrete_map={
		"success":"#636EFA",
		"failed":"#EF553B"
		},
)

fig_month_stat_bar.update_layout(barmode='stack')

fig_month_stat_bar.update_yaxes(
	title_text="Percentage (%)"
)

fig_month_stat_bar.update_xaxes(
	title_text="Month",
	tickvals=list(range(1,13)),
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
)

fig_month_stat_bar.show()